# LLM-Based Schema Mapping

Reference: https://medium.com/@hamzaahmad6292/llm-based-schema-mapping-for-automated-crm-integration-c4837d1b1ed5

In [1]:
import os 
import json
import openai
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

True

In [ ]:
class DatasetMapper:
    def __init__(self):
        self.client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        self.rules = self.load_mapping_rules()
        self.dataset:pd.DataFrame

    def load_mapping_rules(self):
        """Load column mapping rules."""
        return {
            "mapping_rules": {
                "id": {
                    "description": "Represents a unique id of the product."
                },
                "transaction_date": {
                    "description": "Represents a date when the transaction happened. The date should be in a standardized format."
                },
                "store_id": {
                    "description": "Represents a unique id which identifies the store at which the products are sold."
                },
                "cat_nm": {
                    "description": "Contains the category name of product sold."
                },
                "sales": {
                    "description": "Gives the total sales for a product at a particular store at a given date. Fractional values are possible since products can be sold in fractional units."
                }
            } 
        }
        
    def describe_dataset(self, df_head: pd.DataFrame):
        """Generate a JSON description of each column in a dataset."""
        system_prompt = """You are a data analysis expert. When given a pandas DataFrame's head output, 
        carefully analyze each column and provide a comprehensive yet concise description. Your response 
        must be a valid JSON object where:
        - Keys are column names
        - Values are detailed descriptions capturing the column's nature, potential meaning, and data characteristics
        - Descriptions should be precise, informative, and max 100 characters long"""

        user_prompt = f"""Analyze the following DataFrame head and provide a JSON description 
        of each column's characteristics and potential meaning:
        {df_head.to_string()}
        Format your response as a valid JSON object with descriptive insights for each column."""

        try:
            response = self.client.chat.completions.create(
                model = 'gpt-4o-mini',
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.95,
                max_tokens=2000
            ).choices[0].message.content
            return response
        except Exception as e:
            print(f"Error in describe_dataset: {e}")
            return {}
    
    def check_column_compatibility(self, column_name: str, column_desc: str, dataset: dict):
        """Check if a column from one dataset matches another based on descriptions."""
        system_prompt = """You are an expert in checking compatibility for mapping COLUMNS across different datasets. 
        Your task is to determine the best match for a target column from the source dataset based on their descriptions. 
        
        Your response should follow these rules:
        - Compare the **target column's description** with the **descriptions of all source columns**.
        - If the descriptions of the source columns align with the target column, return the **name of the source column** 
          that best matches the target column.
        - If no source column is a good match for the target column, return "Nothing Compatible".
        - You can choose multiple features if and only if they fit.
        - Do not output anything other than the column names.
        - If there is more than one column, separate them with a comma (',')."""

        user_prompt = f"""The target column and its description are as follows:
        Target Column: {column_name}
        Description: {column_desc}

        Here are the source columns with their descriptions:
        {json.dumps(dataset)}

        Your output should only be the column name(s) or "Nothing Compatible"."""

        try:
            response = self.client.chat.completions.create(
                model = 'gpt-4o-mini',
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=1000
            ).choices[0].message.content
            return response.strip()
        except Exception as e:
            print(f"Error in check_column_compatibility: {e}")
            return "Nothing Compatible"
    
    def get_mapping_values(self, report):
        """Get the best column mappings based on rules."""
        mappings = []
        try:
            for key, value in self.rules.get("mapping_rules", {}).items():
                mappings.append(self.check_column_compatibility(
                    column_name=key,
                    column_desc=value,
                    dataset=report
                ))
        except Exception as e:
            print(f"Error in get_mapping_values: {e}")
        return mappings
    
    def map_dataset(self, df: pd.DataFrame):
        """Map columns from a source dataset to a target dataset based on predefined rules."""
        try:
            report = self.describe_dataset(df_head=df.head())
            mappings = self.get_mapping_values(report)
            new_dataset = pd.DataFrame()
            not_compatible = []
            
            keys = list(self.rules.get("mapping_rules", {}).keys())

            for i, source in enumerate(mappings):
                if source == "Nothing Compatible":
                    not_compatible.append(keys[i])
                elif "," in source:
                    new_dataset[keys[i]] = df[[col.strip() for col in source.split(",")]].agg(
                        lambda x: ' '.join(x.dropna().astype(str)), axis=1
                    )
                else:
                    new_dataset[keys[i]] = df[source.strip()]
            
            self.dataset = new_dataset
            return new_dataset, not_compatible
        except Exception as e:
            print(f"Error in map_dataset: {e}")
            return pd.DataFrame(), []
        
    def store_dataset(self, path="./examples/mapped_dataset.csv"):
        """Store the mapped dataset to a CSV file."""
        self.dataset.to_csv(path)

In [3]:
mapper = DatasetMapper()

# load a raw CSV
df_raw = pd.read_csv("train.csv")

# map it to standardized schema
mapped_df, missing_cols = mapper.map_dataset(df_raw)

print(mapped_df.head())
print("Columns with no match:", missing_cols)

# save it
mapper.store_dataset("mapped_train.csv")


```json
{
    "id": "Unique identifier for each record in the dataset.",
    "date": "Date of the sales record in YYYY-MM-DD format.",
    "store_nbr": "Identifier for the store where the sale occurred.",
    "family": "Product category or family of the item sold.",
    "sales": "Total sales amount for the product on the given date.",
    "onpromotion": "Indicator of whether the item was on promotion (1) or not (0)."
}
```
   id transaction_date  store_id      cat_nm  sales
0   0       2013-01-01         1  AUTOMOTIVE    0.0
1   1       2013-01-01         1   BABY CARE    0.0
2   2       2013-01-01         1      BEAUTY    0.0
3   3       2013-01-01         1   BEVERAGES    0.0
4   4       2013-01-01         1       BOOKS    0.0
Columns with no match: []
